# 02 — Build the Training Dataset

**What:** Walk through the synthesis pipeline, show one mixture being assembled,
verify the invariant, generate a small dataset.
**Prerequisites:** Isolated flute clips + flute-free background music.
**Runtime:** ~5 minutes.  **GPU needed:** No.

In [ ]:
# Point these at your actual data
FLUTE_DIR = 'flute_clips/'
BG_DIR = 'backgrounds/'
OUT_DIR = 'data/flute_dataset'

## 1. Inspect a single track build

In [ ]:
from pathlib import Path
from data_pipeline.audio_utils import SAMPLE_RATE
from data_pipeline.build_dataset import build_track

flute_files = sorted(Path(FLUTE_DIR).rglob('*.wav'))
bg_files = sorted(Path(BG_DIR).rglob('*.wav'))

if flute_files and bg_files:
    stems = build_track(
        flute_file=flute_files[0], bg_file=bg_files[0],
        seg_samples=SAMPLE_RATE * 5, snr_db=3.0,
        augment_flute=False, augment_bg=False,
    )
    diff = (stems['mixture'] - (stems['chinese-flute'] + stems['other'])).abs().max()
    print(f'Invariant: max |diff| = {diff.item():.2e}')
else:
    print('Place flute clips in flute_clips/ and backgrounds in backgrounds/')

## 2. Plot the stems

In [ ]:
import matplotlib.pyplot as plt

if flute_files and bg_files:
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    for ax, (name, wav) in zip(axes, stems.items()):
        ax.plot(wav[0, :2000].numpy(), linewidth=0.5)
        ax.set_title(name)
    axes[-1].set_xlabel('Sample')
    plt.tight_layout()
    plt.show()

## 3. Build the full dataset

In [ ]:
from data_pipeline.build_dataset import build_dataset

if flute_files and bg_files:
    build_dataset(
        flute_dir=Path(FLUTE_DIR), bg_dir=Path(BG_DIR),
        out_dir=Path(OUT_DIR), num_train=50, num_valid=10,
        seg_len=8.0, snr_min=-5.0, snr_max=10.0, seed=42,
    )
    print(f'Dataset built at {OUT_DIR}')
else:
    print('Need flute_clips/ and backgrounds/ directories')

## 4. Verify

In [ ]:
import soundfile as sf
out = Path(OUT_DIR)
if out.exists():
    for split in ('train', 'valid'):
        tracks = sorted((out / split).iterdir())
        print(f'{split}: {len(tracks)} tracks')
        if tracks:
            track = tracks[0]
            mix, _ = sf.read(str(track / 'mixture.wav'))
            flute, _ = sf.read(str(track / 'chinese-flute.wav'))
            other, _ = sf.read(str(track / 'other.wav'))
            print(f'  Invariant: max |diff| = {abs(mix - (flute + other)).max():.2e}')